### Animationen mit `asyncio`


Folgendes Pattern erstellt eine einfache Animation mit `asyncio`.
Eine solche Animation läuft im Hintergrund und blockiert das Notebook nicht.





```python
import asyncio


def animation(canvas, fps=25):
    # some initialization
  
    async def animate():
        '''die eigentliche Animation'''
        while True:
            with hold_canvas(canvas):
                # draw frame
            await asyncio.sleep(1/fps)

    task = asyncio.create_task(animate(), name='my_animation')
    return task
```

Der Aufruf von `animate()` führt den Code im Funktionsbody noch nicht aus.
Statt dessen wird (ähnlich wie bei einer Generator-Funktion) eine
sog. Coroutine zurückzugeben.
Aus dieser Coroutine macht dann `asyncio.create_task` einen Task,
der in den Eventloop von Jupyterlab aufgenommen und gestartet wird.
Optional kann der Task benamst werden.

Jedesmal, wenn `await asyncio.sleep(t)` angetroffen wird,
wird der Task pausiert. Der Eventloop kümmert sich um die anderen Tasks.
Nach t Sekunden, wird der Task wieder fortgesetzt.

```
draw frame 0
     ↓
await asyncio.sleep(1/fps)
     ↓
Eventloop kümmert sich um andere Aufgaben
     ↓
resume (nach 1/fps sec)
     ↓
draw frame 1
     ↓
...
```

Es ist wichtig, `task` zurückzugeben:
- Fehlermeldungen von `animate` werden nicht angezeigt,
  `task.exception()` zeigt diese an.
- `task.cancel()` erlaubt, den Task zu stoppen.

In [ ]:
import math
import asyncio
import widget_helpers as W
from ipycanvas import hold_canvas


def show_tasks():
    for i, task in enumerate(asyncio.all_tasks()):
        print(f'Task {i+1}: {task.get_name()}')
        print(f'  - Status:   {task._state}')
        print(f'  - Coroutine: {task.get_coro()}')
        print('-' * 40)

In [ ]:
show_tasks()

In [ ]:
running = True


def play_animation(canvas, fps=24, T=2):
    async def trace_circle():
        t = 0
        dt = 1/fps
        while running:
            with hold_canvas():
                canvas.clear()
                x = canvas.width / 2 + radius*math.cos(2*math.pi*t/T)
                y = canvas.height / 2 + radius*math.sin(2*math.pi*t/T)
                canvas.fill_circle(x, y, 5)
                await asyncio.sleep(dtx)
            t += dt

    radius = canvas.width/3
    task = asyncio.create_task(trace_circle(), name='trace_circle')
    return task

In [ ]:
mcanvas = W.get_mcanvas(2, 200, 200)
fg, bg = mcanvas
fg.fill_style = 'red'
mcanvas

In [ ]:
running = True
task_1 = play_animation(bg, T=2)
# task_2 = play_animation(fg,  T=5)
task_1

In [ ]:
task_1

In [ ]:
show_tasks()

In [ ]:
running = False  # stop task

### Aufgabe
- Füge Fehler in den Animationscode ein und schaue, wie diese im `task` angezeigt werden.  
  Betrachte auch `task.exception()`.
- Modifiziere obige Animation so, dass die Drehrichtung des Punktes von einer
globalen Variable `direction` abhängt (setze z.B. `t += dt*direction`).
